# Indictrans Multilingual Batch Translation

IndicTrans2 translation workflows with batch processing and multilingual file preparation.

Stage: project experiment.

Uses the [AI4Bharat IndicTrans2 inference workflow](https://github.com/AI4Bharat/IndicTrans2), with file/batch processing experiments. Original explanatory cells are retained below.


# IndicTrans2 HF Inference

We provide an example notebook on how to use our IndicTrans2 models which were originally trained with the fairseq to HuggingFace transformers for inference purpose.


## Setup

Please run the cells below to install the necessary dependencies.


In [ ]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [ ]:
%%capture
%cd /content/IndicTrans2/huggingface_interfaceEngaging in a variety of activities can greatly enhance overall well-being and life satisfaction. Physical activities like hiking, cycling, and swimming not only improve fitness but also offer opportunities to connect with nature and experience adventure. Creative pursuits such as painting, writing, or playing an instrument stimulate the mind and foster self-expression. Social activities, including volunteering, joining clubs, or attending community events, help build meaningful relationships and a sense of belonging. Intellectual activities like reading, solving puzzles, and attending workshops or classes keep the mind sharp and encourage lifelong learning. Balancing these diverse activities can lead to a well-rounded and fulfilling life, promoting both personal growth and enjoyment.

In [ ]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers>=4.33.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransTokenizer
%cd IndicTransTokenizer
!python3 -m pip install --editable ./
%cd ..

**IMPORTANT : Restart your run-time first and then run the cells below.**

## Inference


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from IndicTransTokenizer import IndicProcessor, IndicTransTokenizer

BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None

In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, direction, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = IndicTransTokenizer(direction=direction)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            src=True,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(generated_tokens.detach().cpu().tolist(), src=False)

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

### English to Indic Example


In [ ]:
def convert_to_list_of_strings(input_text):
    # Split the input text by newline characters
    lines = input_text.split('\n')
    lines = lines[1:-1]
    # Return the list of lines
    return lines

# Example input
input_text = """
are the three key actions in the sustainability movement aimed at minimizing waste and conserving resources.
This involves reducing the amount of waste generated in the first place by using fewer resources and consuming less. It encourages people to be mindful of their consumption habits and to make choices that minimize their environmental impact.
Recycling involves collecting materials that would otherwise be considered waste and processing them to create new products.
Reuse encourages finding new purposes for items instead of disposing of them after a single use. This can involve repairing, repurposing, or donating items to extend their lifespan and reduce the need for new products.
Recycle
Reuse
Reduce
Reduce, Reuse, Recycle
"""

# Convert the input text into a list of strings
list_of_strings = convert_to_list_of_strings(input_text)

# Print the list of strings
for line in list_of_strings:
    print(line)


In [ ]:
en_sents = convert_to_list_of_strings("""
The Impact of Artificial Intelligence on Modern Healthcare
Dr. John Doe, Prof. Jane Smith
abstract
Keywords:
Introduction
Literature review
Methodology
RESULTS
CONCLUSION
REFERENECES
""")


In [ ]:

en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"  # ai4bharat/indictrans2-en-indic-dist-200M
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

import pandas as pd
from tabulate import tabulate

# en_sents = [
#     "When I was young, I used to go to the park every day.",
#     "He has many old books, which he inherited from his ancestors.",
#     "I can't figure out how to solve my problem.",
#     "She is very hardworking and intelligent, which is why she got all the good marks.",
#     "We watched a new movie last week, which was very inspiring.",
#     "If you had met me at that time, we would have gone out to eat.",
#     "She went to the market with her sister to buy a new sari.",
#     "Raj told me that he is going to his grandmother's house next month.",
#     "All the kids were having fun at the party and were eating lots of sweets.",
#     "My friend has invited me to his birthday party, and I will give him a gift.",
# ]

src_lang, tgt_lang = "eng_Latn" , "hin_Deva"
hi_translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(en_sents, hi_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{translation}")

# Add underscore line under each pair of sentences
data_with_lines = []
for en, hi in zip(en_sents, hi_translations):
    data_with_lines.append((en, hi))
    data_with_lines.append(("_" * len(en), "_" * len(hi)))  # Add lines for separation

# flush the models to free the GPU memory
del en_indic_tokenizer, en_indic_model

###FILE TRANSLATION

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "hin_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_hindi.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###TELUGU_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "tel_Telu"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_telugu.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###BENGALI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "ben_Beng"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_bengali.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###TAMIL_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "tam_Taml"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_tamil.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###GUJARATI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "guj_Gujr"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_gujarati.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###ASSAMESE_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "asm_Beng"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_assamese.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###KONKANI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "gom_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_konkani.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###MARATHI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "mar_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_marathi.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###KANNADA_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "kan_Knda"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_kannada.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###MALAYALAM_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "mal_Mlym"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_malayalam.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###ODIA_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "ory_Orya"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_odia.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###PUNJABI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "pan_Guru"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_punjabi.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###BODO_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "brx_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_bodo.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###DOGRI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "doi_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_dogri.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###MAITHILI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "mai_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_maithili.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###MANIPURI_MEITEI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "mni_Mtei"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_manipuri_meitei.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###MANIPURI_BENGALI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "mni_Beng"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_manipuri_bengali.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###NEPALI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "npi_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_nepalii.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###SANSKRIT_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "san_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_sanskrit.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###SANTALI_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "sat_Olck"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_santali.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###SINDHI_DEVNAGIRI

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "snd_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_sindhi_devnagiri.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###SINDHI_ARABIC

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "snd_Arab"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_sindhi_arabic.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###URDU_FILE

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "urd_Arab"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_urdu.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###KASHMIRI_ARABIC

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "kas_Arab"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_kashmiri_arabic.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")

###KASHMIRI_BENGALI

In [ ]:
# Initialize the model and tokenizer
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

# Read the input file
with open('/content/rp.txt', 'r', encoding='utf-8') as file:
    en_sents = file.readlines()

# Remove any leading/trailing whitespace and empty lines
en_sents = [sent.strip() for sent in en_sents if sent.strip()]

# Set the source and target languages
src_lang, tgt_lang = "eng_Latn", "kas_Deva"  # Change "hin_Deva" to your desired Indic language code

# Perform the translation
translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# Write the translations to a new file
with open('/content/rp_kashmiri_devnagiri.txt', 'w', encoding='utf-8') as file:
    for translation in translations:
        file.write(translation + '\n')

print("Translation complete. Output written to 'translated_output.txt'")